# MAVEN Hallucination Detection Benchmarks

This notebook runs famous hallucination detection benchmarks (TruthfulQA, HaluEval) to evaluate MAVEN's performance.

**Cost**: ~€0.75 for maximum tier (400 TruthfulQA questions)

**Time**: 1-2 hours depending on tier

---

## 📋 Setup Instructions

1. **Get Together AI API Key**: https://api.together.xyz/signup
2. **Run Step 1** below to set your API key
3. **Run Step 2** to install dependencies
4. **Run Step 3** to choose your benchmark tier
5. **Download results** at the end

---

## Step 1: Set Your Together AI API Key

⚠️ **REQUIRED**: Enter your Together AI API key below

In [ ]:
import os
from google.colab import userdata

# Option 1: Enter API key directly (temporary, only for this session)
TOGETHER_API_KEY = ""  # Paste your key here

# Option 2: Use Colab Secrets (recommended)
# Go to 🔑 icon in left sidebar → Add TOGETHER_API_KEY
# Then uncomment the line below:
# TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

if not TOGETHER_API_KEY:
    print("⚠️ WARNING: API key not set! Please set TOGETHER_API_KEY above.")
else:
    os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
    print("✅ API key set successfully!")

## Step 2: Install MAVEN and Dependencies

This will install MAVEN from PyPI and all required dependencies.

In [ ]:
# Install MAVEN from PyPI
!pip install -q maven-ai

# Install benchmark dependencies
!pip install -q datasets  # For TruthfulQA dataset

print("\n✅ Installation complete!")
print("\nVerifying installation...")
import maven
print(f"MAVEN version: {maven.__version__}")

## Step 3: Download Benchmark Scripts

Download the latest benchmark scripts from GitHub.

In [ ]:
# Clone the MAVEN repository to get benchmark scripts
!git clone https://github.com/rwondo/maven.git /content/maven_repo

# Copy benchmark scripts to working directory
!cp /content/maven_repo/benchmark_*.py /content/
!cp /content/maven_repo/run_all_benchmarks.py /content/

# Create results directory
!mkdir -p /content/benchmarks/results

print("\n✅ Benchmark scripts downloaded!")
!ls -lh /content/benchmark*.py /content/run_all_benchmarks.py

## Step 4: Choose Your Benchmark Tier

Select which tier to run based on your budget:

| Tier | Tests | Cost | Time | Description |
|------|-------|------|------|-------------|
| **minimal** | 30 | ~€0.03 | 5 min | Quick validation |
| **standard** | 60 | ~€0.05 | 10 min | Good for docs |
| **comprehensive** | 160 | ~€0.18 | 30 min | Statistical evidence |
| **maximum** | 460 | ~€0.75 | 1-2 hrs | Large scale (50% of full) |

In [ ]:
# Choose your tier: 'minimal', 'standard', 'comprehensive', or 'maximum'
TIER = 'standard'  # Change this to your preferred tier

print(f"Selected tier: {TIER.upper()}")
print("\nStarting benchmarks...")
print("This may take a while depending on the tier.")
print("Progress will be shown below.\n")

## Step 5: Run the Benchmarks

⏱️ This will take some time. You can:
- Watch progress in real-time
- Close browser (Colab keeps running)
- Come back later to download results

💾 Checkpoints are saved automatically every few questions, so if interrupted, you can resume by re-running this cell.

In [ ]:
import subprocess
import sys
from datetime import datetime

start_time = datetime.now()
print(f"🚀 Starting benchmarks at {start_time.strftime('%H:%M:%S')}")
print("=" * 80)

# Run the benchmark suite
result = subprocess.run(
    [sys.executable, '/content/run_all_benchmarks.py', '--tier', TIER],
    cwd='/content',
    capture_output=False,
    text=True
)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds() / 60

print("\n" + "=" * 80)
print(f"✅ Benchmarks completed at {end_time.strftime('%H:%M:%S')}")
print(f"Total duration: {duration:.1f} minutes")

if result.returncode == 0:
    print("\n🎉 All benchmarks ran successfully!")
else:
    print(f"\n⚠️ Benchmark exited with code {result.returncode}")
    print("Check the output above for errors.")

## Step 6: View Results Summary

Quick summary of the benchmark results.

In [ ]:
import json
from pathlib import Path

# Load combined report
report_file = Path(f'/content/benchmarks/results/combined_benchmark_report_{TIER}.json')

if report_file.exists():
    with open(report_file, 'r') as f:
        report = json.load(f)
    
    print("=" * 80)
    print(f"MAVEN BENCHMARK RESULTS - {TIER.upper()} TIER")
    print("=" * 80)
    print()
    
    # Overall metrics
    if 'overall_metrics' in report:
        metrics = report['overall_metrics']
        print("📊 OVERALL METRICS:")
        print(f"  Total tests run: {metrics.get('total_tests', 0)}")
        print(f"  Hallucinations detected: {metrics.get('total_hallucinations_detected', 0)}")
        print(f"  False positives: {metrics.get('total_false_positives', 0)}")
        print()
    
    # TruthfulQA
    if 'truthfulqa' in report.get('benchmarks', {}):
        tq = report['benchmarks']['truthfulqa']
        print("📚 TruthfulQA Results:")
        print(f"  Questions tested: {tq.get('total_questions', 0)}")
        print(f"  Accuracy: {tq.get('accuracy', 0):.1f}%")
        if tq.get('baseline_untruthful', 0) > 0:
            rate = (tq.get('maven_detected', 0) / tq.get('baseline_untruthful', 1)) * 100
            print(f"  Detection rate: {rate:.1f}%")
        print()
    
    # HaluEval
    if 'halueval' in report.get('benchmarks', {}):
        he = report['benchmarks']['halueval']
        print("🎯 HaluEval Results:")
        print(f"  Samples tested: {he.get('total_samples', 0)}")
        print(f"  Detection rate: {he.get('detection_rate', 0):.1f}%")
        print(f"  Perfect results: {he.get('perfect', 0)}/{he.get('total_samples', 0)}")
        print()
    
    print("=" * 80)
    print(f"\n✅ Full report saved to: {report_file}")
    
else:
    print("❌ No results found. Make sure Step 5 completed successfully.")

## Step 7: Download Results

Download all result files to your computer.

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

# Create a zip file with all results
zip_path = f'/content/maven_benchmark_results_{TIER}.zip'

with zipfile.ZipFile(zip_path, 'w') as zipf:
    results_dir = Path('/content/benchmarks/results')
    for file in results_dir.glob('*.json'):
        if 'checkpoint' not in file.name:  # Skip checkpoint files
            zipf.write(file, file.name)
            print(f"Added: {file.name}")

print(f"\n📦 Created zip file: {zip_path}")
print("\nDownloading...")
files.download(zip_path)
print("\n✅ Download complete! Check your Downloads folder.")

## Optional: View Individual Benchmark Details

Explore specific benchmark results in detail.

In [ ]:
import json
import pandas as pd

# Choose which benchmark to explore: 'truthfulqa', 'halueval', or 'hallucination_reduction'
BENCHMARK = 'truthfulqa'  # Change this

result_file = Path(f'/content/benchmarks/results/{BENCHMARK}_benchmark.json')

if result_file.exists():
    with open(result_file, 'r') as f:
        data = json.load(f)
    
    # Show detailed results as a table
    if 'detailed_results' in data:
        df = pd.DataFrame(data['detailed_results'])
        print(f"\n{BENCHMARK.upper()} - First 10 Results:\n")
        print(df.head(10))
    else:
        print(json.dumps(data, indent=2))
else:
    print(f"❌ File not found: {result_file}")

## 🔧 Troubleshooting

**If benchmarks stop unexpectedly:**
1. Just re-run Step 5 - it will resume from the last checkpoint
2. Check your Together AI API credits
3. Reduce tier to 'minimal' or 'standard'

**If you get API errors:**
1. Check your API key in Step 1
2. Verify you have credits: https://api.together.xyz/settings/billing
3. Try reducing to a lower tier

**To start fresh:**
```python
!rm -rf /content/benchmarks/results/*_checkpoint.json
```

---

## 📚 Resources

- **MAVEN GitHub**: https://github.com/rwondo/maven
- **MAVEN PyPI**: https://pypi.org/project/maven-ai/
- **Together AI**: https://api.together.xyz/
- **TruthfulQA Paper**: https://arxiv.org/abs/2109.07958
- **HaluEval Paper**: https://arxiv.org/abs/2305.11747